# PANdeMaiz — Entrenamiento 1D-CNN para Detección Sísmica

**Propósito:** Entrenar una red neuronal liviana para clasificar ventanas de 4 s
como **Ruido (0)** o **Sismo (1)**, a partir de los espectrogramas generados en
`seismic_dataset_builder_v3.ipynb`.

| Parámetro | Valor |
|-----------|-------|
| Input shape | `(65, 11, 3)` float32 |
| Clases | 0 = Ruido (3 284 muestras) · 1 = Sismo (2 004 muestras) |
| Arquitectura | 1D-CNN (reshape frec→feature, Conv1D × 2) |
| Target tamaño | < 300 KB INT8 TFLite |
| Target recall sismo | ≥ 85 % en test set |
| PGA | No incluido en Fase 2 (el espectrograma lo codifica implícitamente) |

**Salidas generadas:**
- `ML/models/seismic_int8.tflite` — modelo cuantizado INT8 (pesos) / float32 (I/O)
- `ML/models/model.h` — arreglo C para incrustar en firmware ESP32
- `ML/models/norm_constants.h` — MEAN, STD y HANN_POWER para normalización en C++
- `ML/models/training_curves.png` — curvas de pérdida y AUC
- `ML/models/confusion_matrix.png` — matriz de confusión en test set


In [18]:
import os

# Forzar ejecución en CPU.
# Si tienes GPU y quieres usarla, comenta la siguiente línea:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")
print(f"GPUs visibles : {tf.config.list_physical_devices('GPU')}")
print(f"CPUs visibles : {tf.config.list_physical_devices('CPU')}")


TensorFlow 2.21.0
GPUs visibles : []
CPUs visibles : [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Celda 2 — Imports y constantes

Centraliza todos los hiperparámetros del experimento en un único lugar para
facilitar la experimentación sin tocar el resto del notebook.

| Constante | Valor | Descripción |
|-----------|-------|-------------|
| `INPUT_SHAPE` | `(65, 12, 3)` | Forma de cada espectrograma (frec, tiempo, canal) |
| `BATCH_SIZE` | 32 | Muestras por mini-batch |
| `MAX_EPOCHS` | 100 | Límite superior; EarlyStopping cortará antes |
| `LR_INIT` | 1e-3 | Tasa de aprendizaje inicial de Adam |
| `SEED` | 42 | Semilla global para reproducibilidad del split |


In [19]:
import numpy as np
from pathlib import Path
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib
matplotlib.use('Agg')  # backend sin pantalla para nbconvert
import matplotlib.pyplot as plt

RUIDO_DIR   = Path("../Data_Labeling/Dataset/0_Ruido")
SISMO_DIR   = Path("../Data_Labeling/Dataset/1_Sismo")
MODEL_DIR   = Path("models"); MODEL_DIR.mkdir(exist_ok=True)

INPUT_SHAPE = (65, 11, 3)
BATCH_SIZE  = 32
MAX_EPOCHS  = 100
LR_INIT     = 1e-3
SEED        = 42


## Celda 4 — Carga del dataset

Carga todos los archivos `.npy` de ambas clases en arrays NumPy.
El orden de carga es reproducible (`sorted`) pero el split posterior mezcla
con semilla fija, por lo que el orden de carga no afecta el resultado.

**class_weight:** compensa el desbalance 1.64:1 (ruido:sismo) penalizando
más los errores sobre la clase minoritaria (sismo) durante el entrenamiento.
El factor es `n_ruido_train / n_sismo_train` para la clase 1; la clase 0 recibe peso 1.

**Split 70 / 15 / 15 estratificado:** mantiene la proporción 1.64:1 en
train, val y test, evitando que el test set tenga distribución diferente.


In [20]:
def load_dataset(ruido_dir, sismo_dir):
    specs, pgas, labels = [], [], []
    for cls_dir, label in [(Path(ruido_dir), 0), (Path(sismo_dir), 1)]:
        for npy in sorted(cls_dir.glob('*.npy')):
            if '_pga' in npy.name:
                continue
            pga_path = npy.parent / (npy.stem + '_pga.npy')
            if not pga_path.exists():
                continue   # omitir muestras sin PGA companion
            specs.append(np.load(npy))
            pgas.append(np.load(pga_path))
            labels.append(label)
    return (np.stack(specs, dtype=np.float32),
            np.stack(pgas,  dtype=np.float32),
            np.array(labels, dtype=np.int32))

X_spec, X_pga, y = load_dataset(RUIDO_DIR, SISMO_DIR)
print(f"Espectrogramas : {X_spec.shape}  dtype={X_spec.dtype}")
print(f"PGA            : {X_pga.shape}  dtype={X_pga.dtype}")
print(f"  Ruido (0): {np.sum(y==0)}   Sismo (1): {np.sum(y==1)}")

X_sp_trv,  X_sp_test,  X_pg_trv,  X_pg_test,  y_trv,  y_test  = train_test_split(
    X_spec, X_pga, y, test_size=0.15, random_state=SEED, stratify=y)
X_spec_train, X_spec_val, X_pga_train, X_pga_val, y_train, y_val = train_test_split(
    X_sp_trv, X_pg_trv, y_trv, test_size=0.176, random_state=SEED, stratify=y_trv)
X_spec_test, X_pga_test = X_sp_test, X_pg_test

n0, n1 = int(np.sum(y_train==0)), int(np.sum(y_train==1))
class_weight = {0: 1.0, 1: round(n0 / n1, 4)}
print(f"\nTrain={len(y_train)}  Val={len(y_val)}  Test={len(y_test)}")
print(f"class_weight = {class_weight}")

Espectrogramas : (5544, 65, 11, 3)  dtype=float32
PGA            : (5544, 3)  dtype=float32
  Ruido (0): 2920   Sismo (1): 2624

Train=3882  Val=830  Test=832
class_weight = {0: 1.0, 1: 1.1132}


## Celda 6 — Normalización Z-score global

**¿Por qué normalizar?** Los valores raw de `log10(PSD + 1e-12)` oscilan
entre −12 (señal nula) y −2 (señal muy energética). Sin normalización,
la función de pérdida converge lento porque los gradientes son pequeños
fuera del rango [−1, 1].

**¿Por qué Z-score escalar (un solo MEAN y STD)?**
- Solo requiere 2 constantes en el firmware del ESP32 (2 floats = 8 bytes).
- El log-espectrograma tiene distribución aproximadamente normal, así que
  el Z-score global es casi tan efectivo como una normalización por bin.
- Se computa **únicamente sobre el training set** para evitar data leakage:
  si usáramos val o test para calcular la media, estaríamos "viendo" el
  futuro durante el entrenamiento.

**En el ESP32:** `x_norm = (log10(PSD + 1e-12) − MEAN) / STD`  
aplicado a cada uno de los 65 × 12 × 3 = 2 340 valores del espectrograma.


In [21]:
# ── Espectrograma: Z-score escalar sobre train set ───────────────────────────
NORM_MEAN = float(X_spec_train.mean())
NORM_STD  = float(X_spec_train.std())
print(f"NORM_MEAN = {NORM_MEAN:.8f}")
print(f"NORM_STD  = {NORM_STD:.8f}")
print(f"Rango raw  : [{X_spec_train.min():.2f}, {X_spec_train.max():.2f}]")
X_n_tmp = (X_spec_train - NORM_MEAN) / NORM_STD
print(f"Rango norm : [{X_n_tmp.min():.2f}, {X_n_tmp.max():.2f}]")
del X_n_tmp

X_spec_train_n = (X_spec_train - NORM_MEAN) / NORM_STD
X_spec_val_n   = (X_spec_val   - NORM_MEAN) / NORM_STD
X_spec_test_n  = (X_spec_test  - NORM_MEAN) / NORM_STD

# ── PGA: normalización por eje sobre train set ────────────────────────────────
PGA_MEAN = X_pga_train.mean(axis=0)   # shape (3,)
PGA_STD  = X_pga_train.std(axis=0)    # shape (3,)
print(f"\nPGA_MEAN (g) = {PGA_MEAN}")
print(f"PGA_STD  (g) = {PGA_STD}")

X_pga_train_n = (X_pga_train - PGA_MEAN) / PGA_STD
X_pga_val_n   = (X_pga_val   - PGA_MEAN) / PGA_STD
X_pga_test_n  = (X_pga_test  - PGA_MEAN) / PGA_STD

np.savez(MODEL_DIR / "norm_params.npz",
         mean=np.float32(NORM_MEAN), std=np.float32(NORM_STD),
         pga_mean=PGA_MEAN.astype(np.float32),
         pga_std=PGA_STD.astype(np.float32))
print("\nnorm_params.npz guardado")

NORM_MEAN = -7.98922157
NORM_STD  = 1.37405312
Rango raw  : [-12.00, -0.99]
Rango norm : [-2.92, 5.10]

PGA_MEAN (g) = [0.01375112 0.01821116 0.04943195]
PGA_STD  (g) = [0.06148684 0.05301067 0.15822884]

norm_params.npz guardado


## Celda 8 — Arquitectura 1D-CNN

El espectrograma tiene forma `(65 frec, 11 tiempo, 3 canales)`.
Para una Conv1D necesitamos que la dimensión de **secuencia** sea el eje temporal:

```
(65, 11, 3)  →  Permute(2,1,3)  →  (11, 65, 3)  →  Reshape  →  (11, 195)
                                                                      ↑
                                               195 = 65 bins × 3 canales
```

Cada uno de los 11 pasos temporales (~0.33 s) tiene 195 features que
codifican el contenido frecuencial completo de los 3 ejes en ese instante.

| Capa | Salida | Parámetros |
|------|--------|------------|
| Input | (65, 11, 3) | — |
| Permute + Reshape | (11, 195) | 0 |
| Conv1D(32, k=3) + BN + ReLU | (11, 32) | 18 848 |
| MaxPool1D(2) | (5, 32) | 0 |
| Conv1D(64, k=3) + BN + ReLU | (5, 64) | 6 400 |
| MaxPool1D(2) | (2, 64) | 0 |
| GlobalAveragePooling1D | (64,) | 0 |
| Dropout(0.3) | (64,) | 0 |
| Dense(32, relu) | (32,) | 2 080 |
| Dense(1, sigmoid) | (1,) | 33 |
| **Total** | | **~27 500 params** |

FP32 ≈ 110 KB · INT8 ≈ 30 KB — bien por debajo del límite de 300 KB.


In [22]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation,
    MaxPooling2D, GlobalAveragePooling2D, Dropout, Dense, Concatenate
)

def build_model(spec_shape=INPUT_SHAPE):
    # Kernels simetricos (3,3) + MaxPool (2,2): evita DEQUANTIZE v2 que genera
    # TFLite con kernels asimetricos (3,1)/(1,3) y que TensorFlowLite_ESP32 v1
    # no soporta. La mejora de Fase 5 es el fix de adxl345ify, no la arquitectura.
    spec_input = Input(shape=spec_shape, name='spectrogram')
    x = Conv2D(32, (3, 3), padding='same')(spec_input)   # (65, 11, 32)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = MaxPooling2D((2, 2))(x)                          # (32, 5, 32)
    x = Conv2D(64, (3, 3), padding='same')(x)            # (32, 5, 64)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    spec_feat = GlobalAveragePooling2D()(x)              # (64,) --- 2D en TFLite

    pga_input = Input(shape=(3,), name='pga')
    pga_feat  = Dense(8, activation='relu')(pga_input)   # (8,)

    merged = Concatenate()([spec_feat, pga_feat])        # (72,)
    x = Dropout(0.3)(merged)
    x = Dense(32, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=[spec_input, pga_input], outputs=output, name='seismic_dual')

model = build_model()
model.summary()


## Celda 10 — Entrenamiento

**Optimizador Adam:** adapta la tasa de aprendizaje por parámetro;
converge más rápido que SGD puro sin ajuste manual del scheduler.

**Métricas:**
- `accuracy` — orientativa (engañosa con clases desbalanceadas)
- `roc_auc` — área bajo la curva ROC; métrica principal de monitoreo

| Callback | Acción |
|----------|--------|
| `EarlyStopping(patience=15)` | Para si val_roc_auc no mejora 15 épocas |
| `ReduceLROnPlateau(patience=5)` | Divide LR × 0.3 si val_loss estanca |
| `ModelCheckpoint` | Guarda el mejor modelo según val_roc_auc |

`restore_best_weights=True` asegura que al final del entrenamiento el
modelo activo es el del mejor epoch, no el del último.


In [23]:
import tensorflow as tf

model.compile(
    optimizer=keras.optimizers.Adam(LR_INIT),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='roc_auc')]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_roc_auc', patience=20, mode='max', restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=10, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(
        MODEL_DIR / 'best_model.keras', monitor='val_roc_auc', mode='max', save_best_only=True)
]

history = model.fit(
    [X_spec_train_n, X_pga_train_n], y_train,
    validation_data=([X_spec_val_n, X_pga_val_n], y_val),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set(title='Loss', xlabel='epoch'); ax1.legend()
ax2.plot(history.history['roc_auc'], label='train')
ax2.plot(history.history['val_roc_auc'], label='val')
ax2.set(title='ROC-AUC', xlabel='epoch'); ax2.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=100)
plt.close()
print("training_curves.png guardado")

Epoch 1/100
122/122 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.7916 - loss: 0.4483 - roc_auc: 0.8226 - val_accuracy: 0.8108 - val_loss: 0.4295 - val_roc_auc: 0.8546 - learning_rate: 0.0010
Epoch 2/100
122/122 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.7986 - loss: 0.4274 - roc_auc: 0.8267 - val_accuracy: 0.8181 - val_loss: 0.3847 - val_roc_auc: 0.8613 - learning_rate: 0.0010
Epoch 3/100
122/122 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.8011 - loss: 0.4190 - roc_auc: 0.8346 - val_accuracy: 0.8181 - val_loss: 0.3719 - val_roc_auc: 0.8642 - learning_rate: 0.0010
Epoch 4/100
122/122 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.8024 - loss: 0.4167 - roc_auc: 0.8350 - val_accuracy: 0.8181 - val_loss: 0.3753 - val_roc_auc: 0.8704 - learning_rate: 0.0010
Epoch 5/100
122/122 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.7998 - loss: 0.4177 - roc_auc: 0.8377 - val_accuracy: 0.8181 - val_loss: 0.3840 - val_roc_auc: 0.8661 - learning_rate: 0.0010
Epoch 6/100
122/122 ━━━━━━━━━

## Celda 12 — Evaluación en test set

El test set (15% del total, ~794 muestras) no fue visto por el modelo
en ningún momento — ni para entrenar ni para ajustar hiperparámetros.

| Métrica | Objetivo | Razón |
|---------|----------|-------|
| Recall sismo | ≥ 0.85 | Un falso negativo (sismo no detectado) es inaceptable |
| F1 sismo | ≥ 0.80 | Balance precision/recall |
| Accuracy global | > 0.80 | Referencia; el recall sismo tiene prioridad |

Si el recall sismo cae por debajo de 0.80, considerar:
- Bajar el umbral de clasificación de 0.5 a 0.3 (tradeoff: más falsos positivos)
- Aumentar `class_weight[1]` y volver a entrenar
- Agregar PGA como feature adicional (Fase 2b)


In [24]:
y_prob = model.predict([X_spec_test_n, X_pga_test_n], verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("=" * 55)
print(classification_report(y_test, y_pred,
                             target_names=['0_Ruido', '1_Sismo'], digits=4))
print("=" * 55)

disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                               display_labels=['Ruido', 'Sismo'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set (umbral = 0.5)')
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=100)
plt.close()
print("confusion_matrix.png guardado")

              precision    recall  f1-score   support

     0_Ruido     1.0000    0.6324    0.7748       438
     1_Sismo     0.7099    1.0000    0.8303       394

    accuracy                         0.8065       832
   macro avg     0.8550    0.8162    0.8026       832
weighted avg     0.8626    0.8065    0.8011       832

confusion_matrix.png guardado


In [25]:
# CALIBRACIÓN DE THRESHOLD SOBRE VALIDATION SET
# Busca el threshold que maximiza F1_sismo con recall_sismo >= 0.90
from sklearn.metrics import f1_score, recall_score

y_val_prob = model.predict([X_spec_val_n, X_pga_val_n], verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 181)

SEISMIC_THRESHOLD = 0.50   # default
best_f1 = 0.0
for t in thresholds:
    y_t = (y_val_prob > t).astype(int)
    rec = recall_score(y_val, y_t, zero_division=0)
    f1  = f1_score(y_val, y_t, pos_label=1, zero_division=0)
    if rec >= 0.90 and f1 > best_f1:
        best_f1 = f1
        SEISMIC_THRESHOLD = float(t)

print(f"Threshold calibrado: {SEISMIC_THRESHOLD:.3f}  (F1_sismo={best_f1:.4f}, recall>=0.90 garantizado)")

Threshold calibrado: 0.050  (F1_sismo=0.8388, recall>=0.90 garantizado)


## Celda 14 — Exportación a TFLite INT8

**¿Por qué INT8?**
- Los pesos FP32 (4 bytes/valor) se comprimen a INT8 (1 byte) → ~4× menos espacio.
- La inferencia en microcontroladores sin FPU dedicada es más rápida con enteros.
- El ESP32 (Xtensa LX6) no tiene unidad SIMD vectorial, pero TFLite Micro
  usa kernels de multiplicación optimizados para INT8.

**Configuración elegida: pesos INT8, I/O float32**

Se cuantizan los pesos y activaciones internas a INT8, pero la entrada y
salida del modelo permanecen en `float32`. Esto simplifica el firmware:
el ESP32 pasa directamente el array normalizado de floats sin necesitar
calcular manualmente los parámetros de cuantización de entrada
(`scale` y `zero_point`) que varían por capa.

**`representative_dataset`:** TFLite necesita 200–500 muestras reales del
training set para calibrar los rangos de activación de cada capa y asignar
los valores INT8 óptimos. Sin este paso, la cuantización sería estática
y perdería precisión.

| Archivo | Descripción |
|---------|-------------|
| `seismic_fp32.tflite` | Línea base sin cuantizar (referencia) |
| `seismic_int8.tflite` | Modelo listo para ESP32 (< 300 KB esperado) |


In [26]:
import shutil, os

# ── Exportar con Keras 3 model.export() — necesario para TFLite INT8 en TF 2.21 ─
# tf.saved_model.save() deja ReadVariable ops sin congelar → fallo en calibración.
# model.export() congela los pesos como constantes en el grafo.
export_path = str(MODEL_DIR / 'keras_export')
if os.path.exists(export_path):
    shutil.rmtree(export_path)
model.export(export_path)
print(f"Keras export guardado en: {export_path}")

# ── FP32 baseline ──────────────────────────────────────────────────────────────
converter_fp32 = tf.lite.TFLiteConverter.from_saved_model(export_path)
tflite_fp32    = converter_fp32.convert()
(MODEL_DIR / 'seismic_fp32.tflite').write_bytes(tflite_fp32)
print(f"FP32 baseline : {len(tflite_fp32)/1024:.1f} KB")

# Verificar orden de inputs en el FP32 — TFLite puede reordenarlos
fp32_interp = tf.lite.Interpreter(model_content=tflite_fp32)
fp32_interp.allocate_tensors()
fp32_inputs = fp32_interp.get_input_details()
print("Orden de inputs en TFLite FP32:")
for d in fp32_inputs:
    print(f"  idx={d['index']}  name={d['name']}  shape={d['shape']}")

# Construir cada batch en el orden correcto del calibrador (por nombre, no por posición)
def _make_batch(i, input_details):
    batch = []
    for d in input_details:
        if 'pga' in d['name']:
            batch.append(X_pga_train_n[i:i+1].astype(np.float32))
        else:
            batch.append(X_spec_train_n[i:i+1].astype(np.float32))
    return batch

# ── INT8 (pesos + activaciones), I/O permanece float32 ──────────────────────────
def repr_gen():
    for i in range(300):
        yield _make_batch(i, fp32_inputs)

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(export_path)
converter_int8.optimizations             = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset   = repr_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type     = tf.float32
converter_int8.inference_output_type    = tf.float32

tflite_int8 = converter_int8.convert()
(MODEL_DIR / 'seismic_int8.tflite').write_bytes(tflite_int8)
print(f"INT8 cuantizado: {len(tflite_int8)/1024:.1f} KB")
print(f"Compresión     : {len(tflite_fp32)/len(tflite_int8):.1f}×")
ok = '✓' if len(tflite_int8)/1024 < 300 else '✗ SUPERA 300 KB'
print(f"Objetivo < 300 KB : {ok}")

# ── Verificar orden de inputs en el INT8 exportado ─────────────────────────────
interp = tf.lite.Interpreter(model_content=tflite_int8)
interp.allocate_tensors()
print("\nInputs del TFLite INT8 (IMPORTANTE: verificar para firmware):")
for i, t in enumerate(interp.get_input_details()):
    print(f"  input({i}): name={t['name']}  shape={t['shape']}")

# Limpiar export temporal
shutil.rmtree(export_path, ignore_errors=True)

INFO:tensorflow:Assets written to: models/keras_export/assets


INFO:tensorflow:Assets written to: models/keras_export/assets


Saved artifact at 'models/keras_export'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 65, 11, 3), dtype=tf.float32, name='spectrogram'), TensorSpec(shape=(None, 3), dtype=tf.float32, name='pga')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136283496324880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136280285344016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136280285339408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136280285340752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136280285342288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136280285343248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136279836770576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136279836771152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136279836771536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136279836772496: Te

W0000 00:00:1779079125.277463    4979 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1779079125.277613    4979 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1779079125.278153    4979 reader.cc:83] Reading SavedModel from: models/keras_export
I0000 00:00:1779079125.282100    4979 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1779079125.282124    4979 reader.cc:147] Reading SavedModel debug info (if present) from: models/keras_export
I0000 00:00:1779079125.311847    4979 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1779079125.428246    4979 loader.cc:220] Running initialization op on SavedModel bundle at path: models/keras_export
I0000 00:00:1779079125.476821    4979 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 198684 microseconds.
/home/pinpa/Documents/Universidad/Trabajo_de_Grado_Seismology/PANdeMaiz/pan_env/lib/python3.12/site-packages/tensorflow/lite/python/interpreter

FP32 baseline : 102.4 KB
Orden de inputs en TFLite FP32:
  idx=0  name=serving_default_pga:0  shape=[1 3]
  idx=1  name=serving_default_spectrogram:0  shape=[ 1 65 11  3]


W0000 00:00:1779079126.470339    4979 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1779079126.470362    4979 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1779079126.470583    4979 reader.cc:83] Reading SavedModel from: models/keras_export
I0000 00:00:1779079126.472992    4979 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1779079126.473003    4979 reader.cc:147] Reading SavedModel debug info (if present) from: models/keras_export
I0000 00:00:1779079126.494217    4979 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1779079126.597902    4979 loader.cc:220] Running initialization op on SavedModel bundle at path: models/keras_export
I0000 00:00:1779079126.631616    4979 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 161054 microseconds.


INT8 cuantizado: 38.3 KB
Compresión     : 2.7×
Objetivo < 300 KB : ✓

Inputs del TFLite INT8 (IMPORTANTE: verificar para firmware):
  input(0): name=serving_default_pga:0  shape=[1 3]
  input(1): name=serving_default_spectrogram:0  shape=[ 1 65 11  3]


fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1779079127.353370    4979 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


## Comparación de modelos: Keras FP32 vs TFLite FP32 vs TFLite INT8

Tamaños de archivo, correlación de scores de salida y métricas de clasificación sobre el test set.

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ── 1. Tamaños y compresión ───────────────────────────────────────────────────
keras_size = (MODEL_DIR / 'best_model.keras').stat().st_size
fp32_size  = (MODEL_DIR / 'seismic_fp32.tflite').stat().st_size
int8_size  = (MODEL_DIR / 'seismic_int8.tflite').stat().st_size

print("=== Tamaños y compresión (referencia: Keras FP32) ===")
for name, sz in [('Keras FP32', keras_size), ('TFLite FP32', fp32_size), ('TFLite INT8', int8_size)]:
    ratio = keras_size / sz
    print(f"  {name:<14}: {sz/1024:7.1f} KB   {ratio:.2f}× reducción")

# ── 2. Inferencia TFLite en test set ─────────────────────────────────────────
def tflite_predict(tflite_bytes, X_spec, X_pga):
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    pga_idx  = next(d['index'] for d in interp.get_input_details() if 'pga' in d['name'])
    spec_idx = next(d['index'] for d in interp.get_input_details() if 'pga' not in d['name'])
    out_idx  = interp.get_output_details()[0]['index']
    preds = np.empty(len(X_spec), dtype=np.float32)
    for i in range(len(X_spec)):
        interp.set_tensor(pga_idx,  X_pga[i:i+1].astype(np.float32))
        interp.set_tensor(spec_idx, X_spec[i:i+1].astype(np.float32))
        interp.invoke()
        preds[i] = interp.get_tensor(out_idx)[0, 0]
    return preds

print(f"\nInferencia en test set (n={len(y_test)}, threshold={SEISMIC_THRESHOLD:.2f}) ...")
y_prob_keras = model.predict([X_spec_test_n, X_pga_test_n], verbose=0).ravel()
y_prob_fp32  = tflite_predict((MODEL_DIR/'seismic_fp32.tflite').read_bytes(), X_spec_test_n, X_pga_test_n)
y_prob_int8  = tflite_predict((MODEL_DIR/'seismic_int8.tflite').read_bytes(), X_spec_test_n, X_pga_test_n)
print("Inferencia completada.")

# ── 3. Métricas ───────────────────────────────────────────────────────────────
model_names = ['Keras FP32', 'TFLite FP32', 'TFLite INT8']
model_probs = [y_prob_keras, y_prob_fp32, y_prob_int8]
model_sizes = [keras_size, fp32_size, int8_size]

metrics = []
print("\n=== Métricas por modelo (test set, clase Sismo) ===")
for name, probs in zip(model_names, model_probs):
    preds = (probs > SEISMIC_THRESHOLD).astype(int)
    m = dict(
        name=name,
        precision=precision_score(y_test, preds, pos_label=1, zero_division=0),
        recall=recall_score(y_test, preds, pos_label=1, zero_division=0),
        f1=f1_score(y_test, preds, pos_label=1, zero_division=0),
        auc=roc_auc_score(y_test, probs)
    )
    metrics.append(m)
    print(f"  {name:<14}  Prec={m['precision']:.3f}  Rec={m['recall']:.3f}  F1={m['f1']:.3f}  AUC={m['auc']:.3f}")

# ── 4. Visualización ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Barras de métricas
keys    = ['precision', 'recall', 'f1', 'auc']
klabels = ['Precision\nSismo', 'Recall\nSismo', 'F1\nSismo', 'ROC-AUC']
colors  = ['steelblue', 'tomato', 'mediumseagreen', 'goldenrod']
x = np.arange(len(model_names)); w = 0.18
for j, (key, lbl, col) in enumerate(zip(keys, klabels, colors)):
    vals = [m[key] for m in metrics]
    axes[0].bar(x + j*w - 1.5*w, vals, w, label=lbl, color=col, alpha=0.85)
    for xi, v in zip(x + j*w - 1.5*w, vals):
        axes[0].text(xi, v+0.012, f'{v:.2f}', ha='center', fontsize=7.5, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels([m['name'].replace(' ','\n') for m in metrics], fontsize=10)
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel('Score')
axes[0].set_title('Métricas de clasificación — Clase Sismo', fontsize=11)
axes[0].legend(loc='lower right', fontsize=8); axes[0].yaxis.grid(True, alpha=0.35)

# (b) Correlación entre scores
all_probs = np.stack([y_prob_keras, y_prob_fp32, y_prob_int8])
corr = np.corrcoef(all_probs)
short = ['Keras\nFP32', 'TFLite\nFP32', 'TFLite\nINT8']
im = axes[1].imshow(corr, vmin=0.85, vmax=1.0, cmap='YlGn')
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(short, fontsize=10)
axes[1].set_yticks(range(3)); axes[1].set_yticklabels(short, fontsize=10)
axes[1].set_title('Correlación de scores de salida\n(Pearson, test set)', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{corr[i,j]:.4f}', ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1])

# (c) Tamaños de archivo
bar_colors = ['#4472C4', '#ED7D31', '#70AD47']
bar_vals   = [sz/1024 for sz in model_sizes]
bars = axes[2].bar(model_names, bar_vals, color=bar_colors, width=0.45, alpha=0.9)
for bar, sz, kbsz in zip(bars, model_sizes, bar_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, kbsz+1,
                 f'{kbsz:.0f} KB\n({keras_size/sz:.1f}×)', ha='center', fontsize=9.5, fontweight='bold')
axes[2].set_ylabel('Tamaño (KB)')
axes[2].set_title('Tamaño de modelos\n(× = compresión respecto a Keras FP32)', fontsize=11)
axes[2].set_xticklabels([n.replace(' ','\n') for n in model_names], fontsize=10)
axes[2].yaxis.grid(True, alpha=0.35); axes[2].set_ylim(0, max(bar_vals)*1.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'model_comparison.png', dpi=100)
plt.close()
print("\nmodel_comparison.png guardado")

=== Tamaños y compresión (referencia: Keras FP32) ===
  Keras FP32    :   386.3 KB   1.00× reducción
  TFLite FP32   :   102.4 KB   3.77× reducción
  TFLite INT8   :    38.3 KB   10.08× reducción

Inferencia en test set (n=832, threshold=0.05) ...
Inferencia completada.

=== Métricas por modelo (test set, clase Sismo) ===
  Keras FP32      Prec=0.710  Rec=1.000  F1=0.830  AUC=0.832
  TFLite FP32     Prec=0.710  Rec=1.000  F1=0.830  AUC=0.832
  TFLite INT8     Prec=0.710  Rec=1.000  F1=0.830  AUC=0.842


/tmp/ipykernel_4979/3962930958.py:93: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[2].set_xticklabels([n.replace(' ','\n') for n in model_names], fontsize=10)



model_comparison.png guardado


## Celda 16 — Cabeceras C para el firmware ESP32

Esta celda produce dos archivos `.h` listos para copiar a
`platform_acel/acel_ubuntu/src/`:

**`model.h`**  
Convierte `seismic_int8.tflite` a un arreglo C de bytes (`uint8_t[]`).
El ESP32 lo carga en flash (PROGMEM) y TFLite Micro lo interpreta en
tiempo de ejecución sin necesidad de sistema de archivos.

**`norm_constants.h`**  
Exporta las constantes de normalización calculadas en la Celda 7:
- `SEISMIC_NORM_MEAN` / `SEISMIC_NORM_STD` — para el Z-score en C++
- `SEISMIC_HANN_POWER` — suma de cuadrados de la ventana Hanning de 128
  puntos (idéntica a la usada por scipy), necesaria para que el PSD
  calculado en el ESP32 coincida numéricamente con el del entrenamiento.

> ⚠️ Si se re-entrena el modelo (nuevas épocas, más datos, hiperparámetros
> distintos), se deben regenerar y copiar **ambas** cabeceras. Las constantes
> de normalización cambian si el dataset de entrenamiento cambia.


In [28]:
import scipy.signal as signal

# Constante de ventana Hanning — idéntica a scipy.signal.get_window('hann', 128)
win        = signal.get_window('hann', 128)
HANN_POWER = float(np.sum(win ** 2))
print(f"HANN_POWER = {HANN_POWER:.8f}  (Σhann² de 128 puntos)")

# ── model.h ──────────────────────────────────────────────────────────────────
data   = (MODEL_DIR / 'seismic_int8.tflite').read_bytes()
c_body = ', '.join(f'0x{b:02x}' for b in data)
(MODEL_DIR / 'model.h').write_text(
    f'#pragma once\n'
    f'// Auto-generado por seismic_cnn_trainer.ipynb — NO editar a mano\n'
    f'// Arquitectura : seismic_dual (spec Conv2D + pga Dense)  |  Cuantizacion : INT8 (I/O float32)\n'
    f'// Tamano       : {len(data)} bytes = {len(data)/1024:.1f} KB\n'
    f'// ATENCION — orden de inputs en TFLite (verificado por get_input_details):\n'
    f'//   input(0) : (1, 3)         float32  — PGA normalizado por eje\n'
    f'//   input(1) : (1, 65, 11, 3) float32  — espectrograma normalizado\n'
    f'//   output   : float32 score en [0=ruido, 1=sismo]\n'
    f'#include <cstdint>\n\n'
    f'const uint8_t  seismic_model_data[] = {{{c_body}}};\n'
    f'const uint32_t seismic_model_len    = {len(data)}u;\n',
    encoding='utf-8'
)
print(f"model.h generado ({len(data)/1024:.1f} KB)")

# ── norm_constants.h ─────────────────────────────────────────────────────────
(MODEL_DIR / 'norm_constants.h').write_text(
    f'#pragma once\n'
    f'// Auto-generado por seismic_cnn_trainer.ipynb — NO editar a mano\n'
    f'// Aplicar en ESP32 ANTES de pasar el espectrograma al modelo:\n'
    f'//   x_norm = (log10(PSD + 1e-12) - MEAN) / STD\n\n'
    f'// Z-score global (calculado sobre el training set)\n'
    f'constexpr float SEISMIC_NORM_MEAN  = {NORM_MEAN:.8f}f;\n'
    f'constexpr float SEISMIC_NORM_STD   = {NORM_STD:.8f}f;\n\n'
    f'// Ventana Hanning 128 puntos — scipy.signal.get_window("hann", 128)\n'
    f'// PSD = |FFT|^2 / (FS * HANN_POWER)   (one-sided, igual que scipy)\n'
    f'constexpr float SEISMIC_HANN_POWER = {HANN_POWER:.8f}f;\n'
    f'constexpr float SEISMIC_FS         = 200.0f;\n\n'
    f'// Umbral de clasificacion (score > THRESHOLD -> sismo)\n'
    f'// Calibrado sobre validation set: maximiza F1_sismo con recall>=0.90\n'
    f'constexpr float SEISMIC_THRESHOLD  = {SEISMIC_THRESHOLD:.2f}f;\n\n'
    f'// PGA normalization — por eje, calculado sobre train set\n'
    f'// Generado por seismic_cnn_trainer.ipynb; reemplazar tras re-entrenar.\n'
    f'// Aplicar: pga_norm[c] = (pga_g[c] - MEAN[c]) / STD[c]\n'
    f'constexpr float SEISMIC_PGA_MEAN[3] = {{{PGA_MEAN[0]:.8f}f, {PGA_MEAN[1]:.8f}f, {PGA_MEAN[2]:.8f}f}};\n'
    f'constexpr float SEISMIC_PGA_STD[3]  = {{{PGA_STD[0]:.8f}f,  {PGA_STD[1]:.8f}f,  {PGA_STD[2]:.8f}f}};\n',
    encoding='utf-8'
)
print("norm_constants.h generado")
print(f"\n-> Copiar ambos archivos a: platform_acel/acel_ubuntu/src/")

HANN_POWER = 48.00000000  (Σhann² de 128 puntos)
model.h generado (38.3 KB)
norm_constants.h generado

-> Copiar ambos archivos a: platform_acel/acel_ubuntu/src/
